|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>Sampling<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: a batched sampler you can prove is correct<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

Write the batched sampler, with per-request seeds and a test that can
actually catch a wrong one.

This is stage 13. The performance part you already saw; this is about being
right, which is harder to check because a broken sampler writes perfectly
good English.

In [2]:
### run this cell

dev   = 'cuda' if torch.cuda.is_available() else 'cpu'
VOCAB = 151936
torch.manual_seed(0)

def batch(B):
  return (torch.randn(B, VOCAB, device=dev),
          torch.rand(B, device=dev)*1.5 + 0.2,      # temperature
          torch.rand(B, device=dev)*0.3 + 0.7)      # top_p

print(f'vocabulary {VOCAB:,}')

vocabulary 151,936


# Exercise 1: one pass, one seed per request

`torch.multinomial` draws from a single global generator, so a batch is not
independently reproducible. Do the inverse-CDF draw yourself.

In [3]:
def sample(logits, temps, top_ps, seeds, K=64):
  lg = logits / temps[:, None]
  srt, idx = torch.topk(lg, K, dim=-1)
  pr  = F.softmax(srt, dim=-1)
  cum = pr.cumsum(dim=-1)
  pr  = pr * ((cum - pr) < top_ps[:, None])
  pr  = pr / pr.sum(dim=-1, keepdim=True)

  # one uniform per row, drawn from that row's own generator
  u = torch.stack([
        torch.rand(1, generator=torch.Generator(device=logits.device).manual_seed(int(s)),
                   device=logits.device)[0]
        for s in seeds])
  picked = (pr.cumsum(-1) < u[:, None]).sum(-1).clamp(max=K-1)
  return idx.gather(1, picked[:, None]).squeeze(1)

lg, t, p = batch(8)
seeds = torch.arange(8)
a = sample(lg, t, p, seeds)
b = sample(lg, t, p, seeds)
print('same seeds give the same tokens:', torch.equal(a, b))
print('different seeds differ:         ',
      not torch.equal(a, sample(lg, t, p, seeds + 100)))

same seeds give the same tokens: True
different seeds differ:          True


# Exercise 2: does it sample the right distribution?

Four tokens, known probabilities, twenty thousand draws. The histogram is the
test.

In [4]:
# a distribution we know exactly
probs = torch.tensor([0.5, 0.3, 0.15, 0.05], device=dev)
logits = probs.log()[None, :].repeat(20000, 1)
temps  = torch.ones(20000, device=dev)
tops   = torch.ones(20000, device=dev)        # top_p = 1: no truncation
seeds  = torch.arange(20000)

drawn = sample(logits, temps, tops, seeds, K=4)
counts = torch.bincount(drawn, minlength=4).float()
empirical = counts / counts.sum()

print(f"{'token':>6} {'wanted':>8} {'got':>8}")
for i,(w,g) in enumerate(zip(probs.tolist(), empirical.tolist())):
  print(f'{i:>6} {w:>8.3f} {g:>8.3f}')
print(f'\nmax error {(empirical-probs).abs().max():.4f}')

 token   wanted      got
     0    0.500    0.504
     1    0.300    0.296
     2    0.150    0.150
     3    0.050    0.050

max error 0.0038


# Exercise 3: is top-p exact?

Work out on paper which tokens `top_p = 0.9` keeps and what the renormalised
distribution over them is. Then check that is what you drew.

In [5]:
target = 0.9
logits = probs.log()[None, :].repeat(20000, 1)
tops   = torch.full((20000,), target, device=dev)

drawn  = sample(logits, temps, tops, seeds, K=4)
counts = torch.bincount(drawn, minlength=4).float()
emp    = counts/counts.sum()

# by hand: top_p=0.9 keeps 0.5 and 0.3 and 0.15 (0.5, 0.8, 0.95 cumulative)
kept = probs.clone()
cum  = probs.cumsum(0)
kept[(cum - probs) >= target] = 0
kept = kept/kept.sum()

print(f"{'token':>6} {'exact':>8} {'sampled':>9}")
for i,(w,g) in enumerate(zip(kept.tolist(), emp.tolist())):
  print(f'{i:>6} {w:>8.3f} {g:>9.3f}')
print(f'\nmax error {(emp-kept).abs().max():.4f}')

 token    exact   sampled
     0    0.526     0.530
     1    0.316     0.312
     2    0.158     0.157
     3    0.000     0.000

max error 0.0039


### What the distributional test is for

A sampler that is subtly wrong still produces fluent text. Off-by-one in
the top-p boundary, a missing renormalisation, the wrong comparison
operator: none of them crash, none of them look wrong in a demo, and all
of them change what your model is.

So test it the only way that can catch that: draw twenty thousand times
from a distribution you computed by hand, and compare the histogram.

Two boundaries worth arguing about in your own implementation:

- **Does top-p include the token that crosses p?** `(cum - pr) < p`
  includes it; `cum < p` does not. With p = 0.9 on the distribution
  above, one choice keeps three tokens and the other keeps two.
- **Is temperature applied before or after top-p?** Before, always, or
  top-p is measuring mass that temperature is about to move.

### And the seeds

Per-request seeding is the reason a support ticket saying "it gave me
this yesterday" can be answered. It is also awkward in torch, because
the RNG is global and the batch is not.

On the JAX track this is free. JAX has no global RNG at all: a key is a
value you carry with the request, split when you use it, and the same
key always gives the same draw. Stage 13 is much shorter over there, and
the reason is a design decision made years earlier for unrelated
reasons.

    ./vc guide 13